# LLM-GQP Benchmark Report

Comprehensive analysis of all benchmark experiments launched via `benchmark.sh`.
Re-run this notebook after adding new results — every path is derived dynamically.

## 1. Setup

Import libraries and locate the repository root by searching for `benchmark.sh`.
All downstream paths are derived from `REPO_ROOT` — no hardcoded locations.

In [43]:
import os, sys, re
from pathlib import Path
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

for _p in [Path.cwd(), Path.cwd().parent]:
    if (_p / 'benchmark.sh').exists():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError('benchmark.sh not found — run kernel from repo root or notebooks/')

RESULTS_DIR = REPO_ROOT / 'results'
sys.path.insert(0, str(REPO_ROOT))
print(f'Repo root : {REPO_ROOT}')
print(f'Results   : {RESULTS_DIR}')

Repo root : /home/qamar/Desktop/llm-gqp
Results   : /home/qamar/Desktop/llm-gqp/results


## 2. Scan Result Directories

Walk `results/` and parse every run directory using the known naming convention.
Each directory encodes the graph name, model, and shot count in its path.

In [44]:
GRAPH_RE = re.compile(
    r'^erdos_renyi_(?P<direction>directed|undirected)'
    r'_(?P<weight>weighted|unweighted)'
    r'_(?P<size>\w+)_seed(?P<seed>\d+)$'
)
RUN_RE = re.compile(r'^(?P<safe_model>.+)_(?P<shots>\d+)shot$')

def detect_provider(safe_model: str) -> str:
    if re.match(r'gpt-|^o[13]-', safe_model):  return 'openai'
    if re.match(r'gemini',        safe_model):  return 'gemini'
    return 'ollama'

runs = []
for graph_dir in sorted(RESULTS_DIR.iterdir()):
    if not graph_dir.is_dir() or graph_dir.name.startswith('_'):
        continue
    gm = GRAPH_RE.match(graph_dir.name)
    if not gm:
        continue
    gmeta = {**gm.groupdict(), 'graph_name': graph_dir.name}

    for run_dir in sorted(graph_dir.iterdir()):
        if not run_dir.is_dir():
            continue
        rm = RUN_RE.match(run_dir.name)
        if not rm:
            continue
        safe_model = rm.group('safe_model')
        shots = int(rm.group('shots'))
        csv_path = run_dir / f'{run_dir.name}.csv'

        if csv_path.exists():
            try:
                n_rows = sum(1 for _ in open(csv_path)) - 1
                status = 'complete' if n_rows >= 40 else 'partial'
            except Exception:
                n_rows, status = 0, 'error'
        else:
            n_rows, status = 0, 'empty'

        runs.append({
            **gmeta,
            'safe_model': safe_model,
            'provider': detect_provider(safe_model),
            'shots': shots,
            'csv_path': str(csv_path) if csv_path.exists() else None,
            'status': status,
            'n_rows': n_rows,
        })

runs_df = pd.DataFrame(runs)
print(f'Found {len(runs_df)} run directories across {runs_df["graph_name"].nunique()} graphs')

Found 25 run directories across 4 graphs


## 3. Load CSV Data

Read every available CSV into a single master DataFrame with metadata columns added.
Metadata (model, shots, direction, weight) comes from the parsed directory names.

In [45]:
dfs = []
for _, row in runs_df[runs_df['csv_path'].notna()].iterrows():
    try:
        df = pd.read_csv(row['csv_path'])
        for col in ['graph_name', 'safe_model', 'provider', 'shots',
                    'direction', 'weight', 'size', 'seed']:
            df[col] = row[col]
        dfs.append(df)
    except Exception as e:
        print(f'[WARN] {row["csv_path"]}: {e}')

if dfs:
    data = pd.concat(dfs, ignore_index=True)
    data['shots'] = data['shots'].astype(int)
    data['correct'] = data['correct'].map(
        {True: True, False: False, 'True': True, 'False': False}
    )
    print(f'Loaded {len(data):,} rows from {len(dfs)} CSV files')
    print(f'Scoreable rows (correct not null): {data["correct"].notna().sum():,}')
else:
    data = pd.DataFrame()
    print('No CSV data found.')

Loaded 720 rows from 16 CSV files
Scoreable rows (correct not null): 720


## 4. Benchmark Settings Detected

Print the benchmark configuration inferred from the directory structure and loaded data.
This summarises what was actually launched, not what is configured in benchmark.sh.

In [46]:
print('=' * 60)
print('BENCHMARK SETTINGS  (detected from results/)')
print('=' * 60)
print(f'Models       : {sorted(runs_df["safe_model"].unique())}')
print(f'Providers    : {sorted(runs_df["provider"].unique())}')
print(f'Shot counts  : {sorted(runs_df["shots"].unique())}')
print(f'Graph dirs   : {sorted(runs_df["graph_name"].unique())}')
print(f'Directions   : {sorted(runs_df["direction"].unique())}')
print(f'Weights      : {sorted(runs_df["weight"].unique())}')
print(f'Sizes        : {sorted(runs_df["size"].unique())}')
print(f'Seeds        : {sorted(runs_df["seed"].unique())}')
if not data.empty:
    print(f'Descriptors  : {sorted(data["format"].unique())}')
    print(f'Problems     : {sorted(data["problem"].unique())}')
    print(f'Total rows   : {len(data):,}')

BENCHMARK SETTINGS  (detected from results/)
Models       : ['gpt-4o-mini', 'graphwalker', 'graphwalker_latest', 'llama3.1_8b']
Providers    : ['ollama', 'openai']
Shot counts  : [np.int64(0), np.int64(2)]
Graph dirs   : ['erdos_renyi_directed_unweighted_small_seed1', 'erdos_renyi_directed_weighted_small_seed1', 'erdos_renyi_undirected_unweighted_small_seed1', 'erdos_renyi_undirected_weighted_small_seed1']
Directions   : ['directed', 'undirected']
Weights      : ['unweighted', 'weighted']
Sizes        : ['small']
Seeds        : ['1']
Descriptors  : ['adjacency_list', 'edge_list', 'gml', 'l2sp_paths', 'random_walk']
Problems     : ['clustering', 'cycle', 'degree', 'density', 'edge_count', 'is_connected', 'neighbors', 'node_count', 'shortest_path']
Total rows   : 720


## 5. Experiment Inventory

List every discovered run directory with its completion status and row count.
Status: ✅ complete (≥40 rows) | ⚠️ partial | ❌ empty (dir exists, no CSV) | 🔴 error.

In [47]:
STATUS_ICON = {'complete': '✅', 'partial': '⚠️', 'empty': '❌', 'error': '🔴'}

inv = runs_df[['graph_name', 'safe_model', 'provider', 'shots', 'status', 'n_rows']].copy()
inv['status'] = inv['status'].map(lambda s: f"{STATUS_ICON.get(s, '?')} {s}")
inv = inv.rename(columns={
    'graph_name': 'Graph', 'safe_model': 'Model', 'provider': 'Provider',
    'shots': 'Shots', 'status': 'Status', 'n_rows': 'Rows'
})
inv = inv.sort_values(['Graph', 'Model', 'Shots']).reset_index(drop=True)
display(inv)

,Graph,Model,Provider,Shots,Status,Rows
0,erdos_renyi_directed_unweighted_small_seed1,gpt-4o-mini,openai,0,✅ complete,45
1,erdos_renyi_directed_unweighted_small_seed1,gpt-4o-mini,openai,2,✅ complete,45
2,erdos_renyi_directed_unweighted_small_seed1,graphwalker,ollama,0,❌ empty,0
3,erdos_renyi_directed_unweighted_small_seed1,graphwalker,ollama,2,❌ empty,0
4,erdos_renyi_directed_unweighted_small_seed1,llama3.1_8b,ollama,0,✅ complete,45
5,erdos_renyi_directed_unweighted_small_seed1,llama3.1_8b,ollama,2,✅ complete,45
6,erdos_renyi_directed_weighted_small_seed1,gpt-4o-mini,openai,0,✅ complete,45
7,erdos_renyi_directed_weighted_small_seed1,gpt-4o-mini,openai,2,✅ complete,45
8,erdos_renyi_directed_weighted_small_seed1,graphwalker,ollama,0,❌ empty,0
9,erdos_renyi_directed_weighted_small_seed1,graphwalker,ollama,2,❌ empty,0


## 6. Completion Summary

Count and visualise runs by status across all models and configurations.
Helps identify at a glance which combinations still need to be run.

In [48]:
counts = runs_df['status'].value_counts().rename_axis('status').reset_index(name='count')
print('Run status breakdown:')
display(counts)

COLORS = {'complete': '#4caf50', 'partial': '#ff9800', 'empty': '#f44336', 'error': '#9c27b0'}
vc = runs_df['status'].value_counts()
fig, ax = plt.subplots(figsize=(5, max(2, len(vc) * 0.6)))
bars = ax.barh(vc.index, vc.values,
               color=[COLORS.get(s, '#9e9e9e') for s in vc.index])
for bar, val in zip(bars, vc.values):
    ax.text(val + 0.1, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=9)
ax.set_xlabel('Count')
ax.set_title('Run Status Summary')
plt.tight_layout()
plt.show()

Run status breakdown:


,status,count
0,complete,16
1,empty,9


/tmp/ipykernel_4637/4011338329.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Aggregate Accuracy by Model

Compute overall accuracy (% correct) for each model across all graphs and descriptors.
Rows where the LLM answer could not be parsed are excluded from the denominator.

In [49]:
if data.empty:
    print('No data loaded.')
else:
    acc_model = (
        data.dropna(subset=['correct'])
        .groupby('safe_model')['correct']
        .agg(correct='sum', total='count')
        .assign(accuracy_pct=lambda d: (d['correct'] / d['total'] * 100).round(2))
        .sort_values('accuracy_pct', ascending=False)
        .reset_index()
    )
    display(acc_model)

    fig, ax = plt.subplots(figsize=(7, max(2, len(acc_model) * 0.7)))
    ax.barh(acc_model['safe_model'], acc_model['accuracy_pct'],
            color='#5c85d6', edgecolor='white')
    for i, v in enumerate(acc_model['accuracy_pct']):
        ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)
    ax.set_xlabel('Accuracy (%)')
    ax.set_xlim(0, 108)
    ax.set_title('Overall Accuracy by Model')
    plt.tight_layout()
    plt.show()

,safe_model,correct,total,accuracy_pct
0,gpt-4o-mini,180,360,50.0
1,llama3.1_8b,144,360,40.0


/tmp/ipykernel_4637/2405151944.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Accuracy by Graph Descriptor

Break down accuracy by the GDL format used to represent the graph to the LLM.
Identifies which description styles models handle best across all problems.

In [50]:
if data.empty:
    print('No data loaded.')
else:
    acc_fmt = (
        data.dropna(subset=['correct'])
        .groupby('format')['correct']
        .agg(correct='sum', total='count')
        .assign(accuracy_pct=lambda d: (d['correct'] / d['total'] * 100).round(2))
        .sort_values('accuracy_pct', ascending=False)
        .reset_index()
    )
    display(acc_fmt)

    fig, ax = plt.subplots(figsize=(7, max(2, len(acc_fmt) * 0.6)))
    ax.barh(acc_fmt['format'], acc_fmt['accuracy_pct'],
            color='#d67c5c', edgecolor='white')
    for i, v in enumerate(acc_fmt['accuracy_pct']):
        ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)
    ax.set_xlabel('Accuracy (%)')
    ax.set_xlim(0, 108)
    ax.set_title('Accuracy by Descriptor / Format')
    plt.tight_layout()
    plt.show()

,format,correct,total,accuracy_pct
0,adjacency_list,82,144,56.94
1,gml,78,144,54.17
2,edge_list,74,144,51.39
3,random_walk,56,144,38.89
4,l2sp_paths,34,144,23.61


/tmp/ipykernel_4637/1164029015.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Accuracy by Problem

Break down accuracy by graph-reasoning problem, ordered from easiest to hardest.
Ordering follows the PROBLEM_DIFFICULTY weights defined in config.py.

In [51]:
if data.empty:
    print('No data loaded.')
else:
    try:
        from config import PROBLEM_DIFFICULTY
        problem_order = sorted(PROBLEM_DIFFICULTY, key=PROBLEM_DIFFICULTY.get)
    except ImportError:
        problem_order = None

    acc_prob = (
        data.dropna(subset=['correct'])
        .groupby('problem')['correct']
        .agg(correct='sum', total='count')
        .assign(accuracy_pct=lambda d: (d['correct'] / d['total'] * 100).round(2))
        .reset_index()
    )
    if problem_order:
        order_map = {p: i for i, p in enumerate(problem_order)}
        acc_prob['_ord'] = acc_prob['problem'].map(order_map).fillna(999)
        acc_prob = acc_prob.sort_values('_ord').drop(columns='_ord')
    else:
        acc_prob = acc_prob.sort_values('accuracy_pct', ascending=False)
    display(acc_prob)

    fig, ax = plt.subplots(figsize=(7, max(3, len(acc_prob) * 0.55)))
    ax.barh(acc_prob['problem'], acc_prob['accuracy_pct'],
            color='#5cb85c', edgecolor='white')
    for i, v in enumerate(acc_prob['accuracy_pct']):
        ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)
    ax.set_xlabel('Accuracy (%)')
    ax.set_xlim(0, 108)
    ax.set_title('Accuracy by Problem (easiest → hardest)')
    plt.tight_layout()
    plt.show()

,problem,correct,total,accuracy_pct
7,node_count,66,80,82.50
4,edge_count,14,80,17.50
6,neighbors,48,80,60.00
2,degree,31,80,38.75
8,shortest_path,15,80,18.75
0,clustering,8,80,10.00
3,density,2,80,2.50
5,is_connected,66,80,82.50
1,cycle,74,80,92.50


/tmp/ipykernel_4637/3892639421.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Accuracy by Shot Count

Compare accuracy across different few-shot settings (0-shot, 2-shot, etc.).
Aggregated over all models and graph configurations to show the global effect.

In [52]:
if data.empty:
    print('No data loaded.')
else:
    acc_shots = (
        data.dropna(subset=['correct'])
        .groupby('shots')['correct']
        .agg(correct='sum', total='count')
        .assign(accuracy_pct=lambda d: (d['correct'] / d['total'] * 100).round(2))
        .reset_index()
    )
    display(acc_shots)

    palette = ['#5c85d6', '#d67c5c', '#5cb85c', '#8e6bbf']
    fig, ax = plt.subplots(figsize=(5, 3))
    bars = ax.bar(
        acc_shots['shots'].astype(str), acc_shots['accuracy_pct'],
        color=palette[:len(acc_shots)], edgecolor='white', width=0.4
    )
    ax.set_xlabel('Shots')
    ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 108)
    ax.set_title('Accuracy by Shot Count')
    for bar, v in zip(bars, acc_shots['accuracy_pct']):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 1,
                f'{v:.1f}%', ha='center', fontsize=9)
    plt.tight_layout()
    plt.show()

,shots,correct,total,accuracy_pct
0,0,151,360,41.94
1,2,173,360,48.06


/tmp/ipykernel_4637/2666330480.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Zero-Shot vs Few-Shot Comparison

Pivot accuracy by model and shot count into a side-by-side comparison table.
The delta column shows the accuracy change from adding few-shot examples.

In [53]:
if data.empty or data['shots'].nunique() < 2:
    print('Need at least two shot settings to compare — not enough data.')
else:
    pivot = (
        data.dropna(subset=['correct'])
        .groupby(['safe_model', 'shots'])['correct']
        .mean().mul(100).round(2)
        .unstack('shots')
    )
    pivot.columns = [f'{c}-shot' for c in pivot.columns]
    shot_cols = [c for c in pivot.columns]
    if '0-shot' in pivot.columns and len(shot_cols) > 1:
        last_shot = shot_cols[-1]
        pivot['delta'] = (pivot[last_shot] - pivot['0-shot']).round(2)
        pivot = pivot.rename(columns={'delta': f'delta ({last_shot} - 0-shot)'})
    display(pivot.reset_index().rename(columns={'safe_model': 'Model'}))

,Model,0-shot,2-shot,delta (2-shot - 0-shot)
0,gpt-4o-mini,48.89,51.11,2.22
1,llama3.1_8b,35.00,45.00,10.00


## 12. Accuracy by Graph Type

Break down accuracy by the structural properties of the graph (directed/undirected, weighted/unweighted).
Reveals whether graph topology or edge weights affect LLM reasoning performance.

In [54]:
if data.empty:
    print('No data loaded.')
else:
    acc_type = (
        data.dropna(subset=['correct'])
        .groupby(['direction', 'weight'])['correct']
        .agg(correct='sum', total='count')
        .assign(accuracy_pct=lambda d: (d['correct'] / d['total'] * 100).round(2))
        .reset_index()
    )
    acc_type['graph_type'] = acc_type['direction'] + ' / ' + acc_type['weight']
    display(acc_type[['graph_type', 'correct', 'total', 'accuracy_pct']])

    fig, ax = plt.subplots(figsize=(6, max(2, len(acc_type) * 0.7)))
    ax.barh(acc_type['graph_type'], acc_type['accuracy_pct'],
            color='#8e6bbf', edgecolor='white')
    for i, v in enumerate(acc_type['accuracy_pct']):
        ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=9)
    ax.set_xlabel('Accuracy (%)')
    ax.set_xlim(0, 108)
    ax.set_title('Accuracy by Graph Type')
    plt.tight_layout()
    plt.show()

,graph_type,correct,total,accuracy_pct
0,directed / unweighted,75,180,41.67
1,directed / weighted,75,180,41.67
2,undirected / unweighted,89,180,49.44
3,undirected / weighted,85,180,47.22


/tmp/ipykernel_4637/3564794974.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Model × Problem Heatmap

Visualise accuracy as a colour-coded heatmap across all model and problem combinations.
Green = high accuracy, red = low — identifies systematic strengths and weaknesses.

In [55]:
if data.empty:
    print('No data loaded.')
else:
    try:
        from config import PROBLEM_DIFFICULTY
        col_order = sorted(PROBLEM_DIFFICULTY, key=PROBLEM_DIFFICULTY.get)
    except ImportError:
        col_order = None

    hm = (
        data.dropna(subset=['correct'])
        .groupby(['safe_model', 'problem'])['correct']
        .mean().mul(100).round(1)
        .unstack('problem')
    )
    if col_order:
        hm = hm[[c for c in col_order if c in hm.columns]]

    fig, ax = plt.subplots(figsize=(max(10, len(hm.columns) * 1.2),
                                    max(3, len(hm) * 0.9)))
    sns.heatmap(
        hm, annot=True, fmt='.0f', cmap='RdYlGn',
        vmin=0, vmax=100, linewidths=0.5, ax=ax,
        cbar_kws={'label': 'Accuracy (%)'},
    )
    ax.set_title('Accuracy (%) — Model x Problem')
    ax.set_xlabel('Problem')
    ax.set_ylabel('Model')
    plt.tight_layout()
    plt.show()

/tmp/ipykernel_4637/1285139467.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 14. Caveats and Data Quality

List missing or incomplete runs, unparseable answers, and other data quality issues.
Use this section to prioritise which experiments to re-run or investigate further.

In [56]:
print('=' * 60)
print('CAVEATS AND DATA QUALITY NOTES')
print('=' * 60)

empty_runs = runs_df[runs_df['status'] == 'empty']
if not empty_runs.empty:
    print(f'\n❌  {len(empty_runs)} run dir(s) exist but have no CSV:')
    for _, r in empty_runs.iterrows():
        print(f'   {r["safe_model"]:30s}  shots={r["shots"]}  {r["graph_name"]}')

partial_runs = runs_df[runs_df['status'] == 'partial']
if not partial_runs.empty:
    print(f'\n⚠️   {len(partial_runs)} run(s) are partial (< 40 rows):')
    for _, r in partial_runs.iterrows():
        print(f'   {r["safe_model"]:30s}  shots={r["shots"]}  {r["graph_name"]}  ({r["n_rows"]} rows)')

if not data.empty:
    n_null = data['correct'].isna().sum()
    pct_null = 100 * n_null / len(data)
    print(f'\n📊 Unparseable answers : {n_null}/{len(data)} ({pct_null:.1f}%) — excluded from accuracy')

    no_ans = (data['answer'] == 'No answer found').sum()
    if no_ans:
        print(f'   of which "No answer found" : {no_ans}')

    zero_models = (
        data.dropna(subset=['correct'])
        .groupby('safe_model')['correct'].mean()
        .pipe(lambda s: s[s == 0].index.tolist())
    )
    for m in zero_models:
        print(f'\n🔴 Model "{m}" has 0% accuracy — check for systematic failures')

print('\n📌 Legacy results in results/_legacy/ are excluded from this analysis.')
print('📌 Re-run this notebook after adding new benchmark results.')

CAVEATS AND DATA QUALITY NOTES

❌  9 run dir(s) exist but have no CSV:
   graphwalker                     shots=0  erdos_renyi_directed_unweighted_small_seed1
   graphwalker                     shots=2  erdos_renyi_directed_unweighted_small_seed1
   graphwalker                     shots=0  erdos_renyi_directed_weighted_small_seed1
   graphwalker                     shots=2  erdos_renyi_directed_weighted_small_seed1
   graphwalker                     shots=0  erdos_renyi_undirected_unweighted_small_seed1
   graphwalker                     shots=2  erdos_renyi_undirected_unweighted_small_seed1
   graphwalker_latest              shots=0  erdos_renyi_undirected_unweighted_small_seed1
   graphwalker                     shots=0  erdos_renyi_undirected_weighted_small_seed1
   graphwalker                     shots=2  erdos_renyi_undirected_weighted_small_seed1

📊 Unparseable answers : 0/720 (0.0%) — excluded from accuracy

📌 Legacy results in results/_legacy/ are excluded from this analysis.
📌